# Methods C and D — Colab GPU training and recovery
Owner: Lingyu Wu. This notebook uses the uploaded working files, never pulls from GitHub, never regenerates splits and never starts a test-set evaluation.

**Beginner instructions / 初学者操作**
1. Upload current `src/*.py`, this Notebook, `splits/splits.json` and `requirements.txt` to Drive `MyDrive/crack-task2/project/crack-width-measurement/`. Upload the dataset as `dataset/raw_dataset.zip`, with `images/` and `masks/` at its root. Do not upload Windows .venv, .git, __pycache__ or smoke weights.
2. Choose Runtime → Change runtime type → GPU. In Controls keep `RUN_MODE="formal"`. Choose `RUN_ACTION="new"` for a fresh run. Execute Setup, Environment, Audit, Configuration, then Benchmark. Review the estimate BEFORE manually executing the Training cell. Batch 8, LR 1e-4, 100 epochs are initial settings, not proven optima.
3. Save the printed Run ID. Every completed epoch persists checkpoint, log, CSV and curves to Drive. Console output shows batches and all 69 validation images.
4. After a runtime restart, repeat Setup → Environment → Audit. Set `RUN_ACTION="resume"` and `RESUME_RUN_ID` to the saved ID. Configuration restores saved parameters; Training continues at the next completed epoch. Never have two active runtimes writing the same Run ID.
5. For inspection only choose `RUN_ACTION="inspect"` with that ID, then run Show latest checkpoint / Show training progress. Inspection does not train. During a running cell, watch its console; a second cell on the same kernel cannot run concurrently.
6. `RUN_MODE="smoke"` is manual only. GPU failure in formal mode stops execution; it never changes mode automatically.
7. The optional threshold/D sections require `RUN_POSTPROCESS=True`. Probabilities are temporary and regenerated after VM loss; all thresholds use 69 complete validation photographs in formal mode. Width selection waits for the shared measurement implementation.

**Recovery boundary:** completed epoch. An interrupted partial epoch is replayed. Local CPU equality is tested with the same versions/seed and num_workers=0; bitwise equality on different GPUs, drivers or kernels is not promised.

**Drive reliability:** complete local serialization → staged Drive copy → SHA256 verification → replacement → verified commit receipt. Drive FUSE does not promise truly atomic remote transactions or cloud durability. Immutable snapshots are retained for fallback; reserve roughly 10 GB for 100 full-model snapshots plus aliases, data and figures. Do not delete snapshots/commits during a run.

## Controls — edit only these fields initially


In [ ]:
import os, sys, json, hashlib, shutil, tempfile, uuid, zipfile, re
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import importlib.util
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
RUN_MODE = "formal"             # "formal" or explicitly "smoke"
RUN_ACTION = "new"              # "new", "resume", "inspect"
RESUME_RUN_ID = ""              # required for resume / inspect
DRIVE_ROOT = Path("/content/drive/MyDrive/crack-task2")
DRIVE_PROJECT = DRIVE_ROOT / "project/crack-width-measurement"
DATA_ARCHIVE = DRIVE_ROOT / "dataset/raw_dataset.zip"
RUNS_DIR = Path("/content/drive/MyDrive/crack-task2/runs")
LOCAL_DATA = Path("/content/crack-data")
RUN_BENCHMARK = True
RUN_POSTPROCESS = False          # switch on after training for threshold/D experiments
assert RUN_MODE in ("formal", "smoke") and RUN_ACTION in ("new", "resume", "inspect")


## Colab Setup — stage current uploaded code and immutable dataset
Helpers below use the standard library before importing src.config. Existing local datasets are reused only if archive/split/file hashes match. Unknown or modified local data is never overwritten. ZIP paths and symlinks are checked before extraction. If staging is interrupted, a new temporary extraction can be attempted; existing raw data is preserved.


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def validate_fixed_data(root, split_path, expected_counts=None):
    expected_counts = expected_counts or {"train":320,"val":69,"test":69}
    split_json = json.loads(Path(split_path).read_text(encoding="utf-8"))
    counts = {key:len(split_json[key]) for key in expected_counts}
    if counts != expected_counts:
        raise ValueError(f"Fixed split counts differ: {counts}")
    flat = sum((split_json[key] for key in expected_counts), [])
    if len(flat) != len(set(flat)):
        raise ValueError("Fixed photograph split overlaps or contains duplicate IDs")
    indexes = []
    for folder in ["images","masks"]:
        paths = [p for p in (Path(root)/folder).iterdir() if p.is_file()]
        if any(p.suffix.lower() not in (".jpg",".jpeg",".png",".bmp",".tif",".tiff") for p in paths):
            raise ValueError(f"Unexpected files in {folder}")
        index = {p.stem:p for p in paths}
        if len(index) != len(paths) or set(index) != set(flat):
            raise ValueError(f"{folder}: expected exactly {len(flat)} unique fixed IDs")
        indexes.append(index)
    return counts


def prepare_dataset(archive, destination, split_path, expected_counts=None):
    archive, destination, split_path = Path(archive), Path(destination), Path(split_path)
    if not archive.is_file() or not split_path.is_file():
        raise FileNotFoundError("Missing dataset archive or fixed splits/splits.json")
    archive_hash, split_hash = sha256_file(archive), sha256_file(split_path)
    if destination.exists():
        marker = destination/".crack_dataset_manifest.json"
        if not marker.is_file():
            raise FileExistsError(f"Unmanaged existing data: {destination}; refusing to overwrite")
        manifest = json.loads(marker.read_text(encoding="utf-8"))
        if manifest["archive_sha256"] != archive_hash or manifest["split_sha256"] != split_hash:
            raise ValueError("Cached archive/split differs; choose another fresh LOCAL_DATA path")
        validate_fixed_data(destination, split_path, expected_counts)
        for name, digest in manifest["files"].items():
            if sha256_file(destination/name) != digest:
                raise ValueError(f"Cached raw data modified: {name}; refusing overwrite")
        print("Reusing verified local data", destination)
        return manifest
    destination.parent.mkdir(parents=True, exist_ok=True)
    local_zip = destination.parent / ("crack-dataset-"+archive_hash+".zip")
    if not local_zip.exists():
        pending = local_zip.with_name(local_zip.name+".pending-"+uuid.uuid4().hex)
        shutil.copy2(archive, pending)
        if sha256_file(pending) != archive_hash:
            raise OSError("Drive-to-local archive copy checksum mismatch")
        os.replace(pending, local_zip)
    elif sha256_file(local_zip) != archive_hash:
        raise ValueError("Local ZIP has been modified; refusing overwrite")
    staging = destination.with_name(destination.name+".extract-"+uuid.uuid4().hex)
    staging.mkdir()
    with zipfile.ZipFile(local_zip) as bundle:
        seen = set()
        for entry in bundle.infolist():
            normalized = entry.filename.replace("\\", "/")
            parts = PurePosixPath(normalized)
            if parts.is_absolute() or ".." in parts.parts or not parts.parts or ":" in parts.parts[0]:
                raise ValueError("Unsafe ZIP path: "+entry.filename)
            if parts.parts[0] not in ("images","masks") or (entry.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("ZIP must contain only images/ and masks/, with no symlinks")
            if not entry.is_dir():
                if len(parts.parts) != 2 or normalized.casefold() in seen:
                    raise ValueError("Nested/duplicate dataset filename: "+entry.filename)
                seen.add(normalized.casefold())
                # Read by ZipInfo: CRC is checked while preserving original bytes.
                target = staging.joinpath(*parts.parts)
                target.parent.mkdir(parents=True, exist_ok=True)
                with bundle.open(entry) as source, target.open("xb") as output:
                    shutil.copyfileobj(source, output)
    counts = validate_fixed_data(staging, split_path, expected_counts)
    files = {p.relative_to(staging).as_posix():sha256_file(p) for p in sorted(staging.rglob("*")) if p.is_file()}
    manifest = {"archive_sha256":archive_hash,"split_sha256":split_hash,"counts":counts,"files":files}
    (staging/".crack_dataset_manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
    if destination.exists():
        raise FileExistsError("Destination appeared during staging; refusing overwrite")
    os.replace(staging,destination)
    print("Prepared", len(files)//2, "unchanged image/mask pairs at", destination)
    return manifest


def stage_uploaded_code(source, cache_parent):
    source, cache_parent = Path(source), Path(cache_parent)
    required = ["src/method_c.py","src/method_d.py","src/config.py","src/data.py","src/utils.py",
                "src/evaluate.py","src/measure.py","src/__init__.py","splits/splits.json","requirements.txt"]
    for name in required:
        if not (source/name).is_file(): raise FileNotFoundError(source/name)
    paths = sorted([*source.glob("src/*.py"),source/"splits/splits.json",source/"requirements.txt"])
    hashes = {p.relative_to(source).as_posix():sha256_file(p) for p in paths}
    bundle_hash = hashlib.sha256(json.dumps(hashes,sort_keys=True).encode()).hexdigest()
    destination = cache_parent/bundle_hash
    if destination.exists():
        if any(not (destination/name).is_file() or sha256_file(destination/name)!=digest for name,digest in hashes.items()):
            raise ValueError("Local code snapshot changed; refusing stale/modified code")
    else:
        cache_parent.mkdir(parents=True,exist_ok=True)
        staging=cache_parent/(bundle_hash+".copy-"+uuid.uuid4().hex);staging.mkdir()
        for name,digest in hashes.items():
            (staging/name).parent.mkdir(parents=True,exist_ok=True)
            shutil.copy2(source/name,staging/name)
            if sha256_file(staging/name)!=digest: raise OSError("Code copy hash mismatch: "+name)
        os.replace(staging,destination)
    return destination,{"bundle_sha256":bundle_hash,"files":hashes,"source":"current uploaded working files, not GitHub HEAD"}


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not DRIVE_PROJECT.is_dir(): raise FileNotFoundError(DRIVE_PROJECT)
    if not DATA_ARCHIVE.is_file(): raise FileNotFoundError(DATA_ARCHIVE)
    PROJECT, CODE_VERSION = stage_uploaded_code(DRIVE_PROJECT, Path("/content/crack-code"))
    DATA_MANIFEST = prepare_dataset(DATA_ARCHIVE,LOCAL_DATA,PROJECT/"splits/splits.json")
    DATA_ROOT = LOCAL_DATA
else:
    # Local smoke testing uses existing data, never modifies/rewrites raw files.
    PROJECT = Path(os.environ.get("CRACK_PROJECT_DIR",Path.cwd()))
    if not (PROJECT/"src").is_dir() and (PROJECT.parent/"src").is_dir(): PROJECT=PROJECT.parent
    DATA_ROOT = Path(os.environ.get("CRACK_DATA_DIR",PROJECT/"data/raw"))
    RUNS_DIR = Path(os.environ.get("CRACK_RUNS_DIR",PROJECT/"eval/method_cd"))
    validate_fixed_data(DATA_ROOT,PROJECT/"splits/splits.json")
    CODE_VERSION={"files":{p.relative_to(PROJECT).as_posix():sha256_file(p) for p in sorted(PROJECT.glob("src/*.py"))}}
    DATA_MANIFEST={"archive_sha256":None,"source":"existing local dataset; not rewritten"}
loaded = sys.modules.get("src.config")
if loaded and (loaded.ROOT.resolve()!=PROJECT.resolve() or loaded.DATA_DIR.resolve()!=DATA_ROOT.resolve()):
    raise RuntimeError("Restart runtime after changing code/data paths; src.config was already imported")
os.environ["CRACK_DATA_DIR"]=str(DATA_ROOT)  # MUST precede the first src import
os.environ.setdefault("MPLCONFIGDIR",str(Path(tempfile.gettempdir())/"crack-matplotlib"))
sys.path.insert(0,str(PROJECT))
print("Project:",PROJECT,"Data:",DATA_ROOT,"Runs:",RUNS_DIR)
print("Uploaded code hashes:",json.dumps(CODE_VERSION,indent=2))


## Environment and mandatory GPU check
Keep Colab's installed GPU PyTorch. This cell never installs torch or uses a CPU wheel index. Only absent non-torch modules are installed; existing versions below project minima stop with a specific message. If a binary import fails, report the printed runtime versions and restart after a minimal repair. Local CPU smoke remains supported only by explicit choice.


In [ ]:
needed={"numpy":"numpy>=1.26","scipy":"scipy>=1.11","skimage":"scikit-image>=0.22","cv2":"opencv-python-headless>=4.9","matplotlib":"matplotlib>=3.8","pandas":"pandas>=2.1","PIL":"Pillow"}
missing=[spec for module,spec in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    import subprocess
    subprocess.check_call([sys.executable,"-m","pip","install",*missing])
if importlib.util.find_spec("torch") is None:
    raise RuntimeError("PyTorch missing. Use a fresh Colab GPU runtime; no automatic CPU installation is permitted")
import numpy as np,pandas as pd,matplotlib.pyplot as plt,torch
from packaging.version import Version
from src import config,data,method_c,method_d,utils
ENVIRONMENT=method_c.environment_info()
print(json.dumps(ENVIRONMENT,indent=2))
minimums={"numpy":"1.26","scipy":"1.11","scikit_image":"0.22","opencv":"4.9"}
for name,minimum in minimums.items():
    if Version(ENVIRONMENT[name])<Version(minimum):
        raise RuntimeError(f"{name} {ENVIRONMENT[name]} < required {minimum}; repair only this dependency, then restart")
assert config.DATA_DIR.resolve()==DATA_ROOT.resolve()
RUNTIME=method_c.check_run_mode(RUN_MODE)  # real GPU tensor operation; formal never falls back
if RUNTIME["device"]=="cpu":torch.set_num_threads(min(4,os.cpu_count() or 1))
print("Loaded Method C:",method_c.__file__,"checkpoint format",method_c.CHECKPOINT_VERSION)


## Data audit — retain original 320/69/69 IDs and full image dimensions
No split generation, no JPEG re-encoding or resizing. Shared labels remain gray >127. The saved split has no verified generation-seed metadata; training seed is 42. All source files remain unchanged.


In [ ]:
from PIL import Image
split = data.load_split()
assert {k:len(v) for k,v in split.items()} == {"train":320,"val":69,"test":69}
flat = sum(split.values(), [])
assert len(set(flat)) == 458
image_paths, mask_paths = data._index(str(config.IMAGE_DIR)), data._index(str(config.MASK_DIR))
assert set(image_paths) == set(mask_paths) == set(flat)
size_counts = {}
for image_id in flat:
    with Image.open(image_paths[image_id]) as im, Image.open(mask_paths[image_id]) as gt:
        # OpenCV's shared loader respects EXIF orientation (e.g. photograph 357).
        def oriented_size(pil_image):
            return pil_image.size[::-1] if pil_image.getexif().get(274, 1) in (5,6,7,8) else pil_image.size
        image_size, mask_size = oriented_size(im), oriented_size(gt)
        assert image_size == mask_size, image_id
        size_counts[str(image_size)] = size_counts.get(str(image_size), 0) + 1
print("Fixed split:", {k:len(v) for k,v in split.items()}, "sizes:", size_counts)


## Configuration — new versus resume
`new` makes a unique Run ID and refuses an existing training directory. `resume` restores the original parameters; do not silently edit architecture, learning rate, batch size, epoch budget or sampler. Package/source mismatches are rejected by the trainer. `inspect` reads existing state without training. Keep the initial formal settings below; benchmark first. CPU smoke uses 2 photographs / 1 validation photograph and is explicitly labelled.


In [ ]:
SMOKE=RUN_MODE=="smoke"
RUN_KIND=RUN_MODE
params={**method_c.DEFAULT_PARAMS,**method_c.FORMAL_INITIAL_PARAMS,"seed":42,"num_workers":0,"bce_weight":0.5,
        "foreground_sample_fraction":0.7,"dataset_archive_sha256":DATA_MANIFEST["archive_sha256"]}
if SMOKE:
    params.update(depth=3,base_channels=16,batch_size=2,learning_rate=1e-3,max_epochs=2,samples_per_epoch=8)
train_ids=split["train"][:2] if SMOKE else split["train"]
val_ids=split["val"][:1] if SMOKE else split["val"]
RUNS_DIR.mkdir(parents=True,exist_ok=True)
if RUN_ACTION=="new":
    if globals().get("ACTIVE_NEW_RUN"):
        raise RuntimeError("A new Run ID was already created in this kernel. Use resume/inspect, or restart to create another")
    RUN_ID=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")+"_"+uuid.uuid4().hex[:8]+"_"+RUN_MODE
    OUT=RUNS_DIR/RUN_ID
    OUT.mkdir(exist_ok=False)
    ACTIVE_NEW_RUN=RUN_ID
    method_c._write_json(OUT/"run_setup.json",{"run_id":RUN_ID,"mode":RUN_MODE,"code":CODE_VERSION,
        "dataset_archive_sha256":DATA_MANIFEST["archive_sha256"],"environment":ENVIRONMENT})
    RESUME_FROM=None
else:
    if not RESUME_RUN_ID or not re.fullmatch(r"[A-Za-z0-9_.-]+",RESUME_RUN_ID) or RESUME_RUN_ID in (".",".."):
        raise ValueError("Set RESUME_RUN_ID to the printed Run ID, without slashes")
    RUN_ID=RESUME_RUN_ID;OUT=RUNS_DIR/RUN_ID
    if not OUT.is_dir():raise FileNotFoundError(OUT)
    RESUME_FROM=OUT/"training/last.pt"
    saved=method_c.read_training_checkpoint(RESUME_FROM)
    if saved["run_kind"]!=RUN_MODE:raise ValueError("RUN_MODE differs from saved run")
    if saved["params"].get("dataset_archive_sha256")!=DATA_MANIFEST["archive_sha256"]:
        raise ValueError("Uploaded dataset archive differs from saved run")
    params=saved["params"]
    train_ids=saved["log"]["train_ids"];val_ids=saved["log"]["val_ids"]
if not SMOKE:
    assert len(train_ids)==320 and len(val_ids)==69
    for key,value in method_c.FORMAL_INITIAL_PARAMS.items():
        if params[key]!=value:raise ValueError(f"Formal initial setting {key} must be {value}; deliberate changes require a recorded new plan")
    assert params["num_workers"]==0
CACHE_DIR=(Path("/content/crack-cache") if IN_COLAB else PROJECT/"eval/method_cd_cache")/RUN_ID
CACHE_DIR.mkdir(parents=True,exist_ok=True)
(OUT/"figures").mkdir(exist_ok=True)
print("SAVE THIS RUN ID:",RUN_ID)
print("ACTION",RUN_ACTION,"MODE",RUN_MODE,"train",len(train_ids),"val",len(val_ids))
print("Persistent output:",OUT,"Regenerable cache:",CACHE_DIR)
print(json.dumps(params,indent=2))


## Show latest checkpoint / Show training progress
Safe to rerun while idle or after reconnect. `completed_epoch` is authoritative; the progress file can describe an interrupted partial epoch. During active training use the training cell's console output.


In [ ]:
if (OUT/"training/commits").is_dir():
    print(json.dumps(method_c.show_training_progress(OUT/"training"),indent=2))
else:
    print("No committed epoch yet. A fresh run will first persist initialized epoch 0.")


## Independent GPU Benchmark — review before Training
Measures three batches after one warm-up, data loading, full-image inference on one validation photograph, peak GPU memory and an epoch estimate. It uses a throwaway model, changes no trained weights, and restores RNG state. Full training still validates ALL 69 images every epoch. Estimates exclude upload/setup and can differ substantially from actual epoch times. If validation or loading dominates, report the measured bottleneck before changing the evaluation plan. No automatic AMP, OOM downsize or validation subsampling occurs.


In [ ]:
if RUN_ACTION!="inspect" and RUN_BENCHMARK:
    benchmark=method_c.benchmark_training(train_ids,val_ids,params,RUN_MODE,batches=3)
    method_c._write_json(OUT/"benchmark.json",benchmark)
    print(json.dumps(benchmark,indent=2))
    print("ESTIMATE ONLY: review bottleneck_hint and estimated validation time before executing Training.")
else:
    print("Benchmark skipped by explicit control or inspection mode")


## Start new formal training / Resume existing formal training
Manually execute this cell AFTER inspecting the benchmark. `RUN_ACTION=new` starts a fresh unique run; `resume` uses that Run ID's verified last commit; `inspect` performs no training. On interruption, rerun from Setup with action resume. Do not merely rerun this cell as new. The epoch budget is the total target, not additional epochs. Existing best weights are retained unless validation Dice strictly improves.

Every epoch saves best/last aliases, immutable snapshot and commit receipt, JSON history, CSV and actual curves to Drive. All snapshots are retained (budget roughly 10 GB for 100 epochs); this avoids deleting the only valid recovery copy. A failed alias upload can be recovered from the newest verified immutable commit. Drive cloud durability cannot be guaranteed.


In [ ]:
if RUN_ACTION=="inspect":
    print("Inspection only: no training started")
else:
    training=method_c.train_unet(train_ids,val_ids,params,OUT/"training",run_kind=RUN_MODE,resume_from=RESUME_FROM)
    print("Status:",training["status"],"best epoch",training["best_epoch"],"best Dice",training["best_val_dice"])


## Show saved training curves
These files also update after EACH committed epoch, independently of this cell or final export. Read training.json for actual Precision/Recall and separate data-loading/training/validation timings. If the latest JSON alias was damaged, resume reconciles it from committed state.


In [ ]:
if (OUT/"training/training.json").is_file():
    training=json.loads((OUT/"training/training.json").read_text(encoding="utf-8"))
    history=pd.DataFrame(training["history"])
    print(history.to_string(index=False))
    for name in ["training_loss","validation_dice"]:
        plot=OUT/f"training/figures/{name}.png"
        if plot.is_file():
            figure,ax=plt.subplots(figsize=(7,4));ax.imshow(plt.imread(plot));ax.axis("off");plt.show();plt.close(figure)


## Optional full-validation threshold selection
Set RUN_POSTPROCESS=True only when ready. Formal mode retains all 69 validation images; no test search. Width results remain unavailable while shared measurement is a placeholder.


In [ ]:
if RUN_POSTPROCESS:
    training=json.loads((OUT/"training/training.json").read_text(encoding="utf-8"))
    net=method_c.load_unet(OUT/"training/best.pt")
    if net is None:raise FileNotFoundError("No best checkpoint yet")
    c_params, threshold_rows = method_c.select_threshold(net, val_ids, params, CACHE_DIR/"probabilities")
    c_params.update(checkpoint_sha256=training["checkpoint_sha256"], run_kind=RUN_KIND,
                    checkpoint_format_version=method_c.CHECKPOINT_VERSION,
                    source_sha256=training["source_sha256"], split_sha256=training["split_sha256"])
    method_c._write_json(OUT/"c_parameters_proposed.json", c_params)
    pd.DataFrame(threshold_rows).to_csv(OUT/"threshold_search.csv", index=False)
    print(pd.DataFrame(threshold_rows).to_string(index=False))
    PARAMETER_EXPORT = OUT/"parameter_export"
    # Use the team serialization interface without overwriting team parameter files.
    previous_params_dir = config.PARAMS_DIR
    try:
        config.PARAMS_DIR = PARAMETER_EXPORT
        if not SMOKE and c_params["selection_complete"]:
            print("Formal C parameters:", utils.save_params("C", c_params))
    finally:
        config.PARAMS_DIR = previous_params_dir
    
    method_c._write_json(OUT/"probabilities_summary.json",json.loads((CACHE_DIR/"probabilities/threshold_search.json").read_text(encoding="utf-8")))
else:
    print("Post-processing disabled; set RUN_POSTPROCESS=True when ready")


## Optional Method D tuning and ablation
Set RUN_POSTPROCESS=True only when ready. Formal mode retains all 69 validation images; no test search. Width results remain unavailable while shared measurement is a placeholder.


In [ ]:
if RUN_POSTPROCESS:
    history=pd.DataFrame(training["history"])
    d_params, d_search = method_d.tune_d(val_ids, CACHE_DIR/"probabilities", c_params["threshold"], OUT/"d_tuning")
    d_params.update(checkpoint_sha256=training["checkpoint_sha256"], run_kind=RUN_KIND,
                    split_sha256=training["split_sha256"], source_sha256=method_c.file_sha256(Path(method_d.__file__)))
    method_c._write_json(OUT/"d_parameters_proposed.json", d_params)
    rows, per_image = method_d.evaluate_variants(val_ids, CACHE_DIR/"probabilities", c_params["threshold"], d_params)
    ablation = pd.DataFrame(rows)
    per_image = pd.DataFrame(per_image)
    per_image.to_csv(OUT/"ablation_per_image.csv", index=False)
    pd.DataFrame(d_search).to_csv(OUT/"d_search.csv", index=False)
    ablation.to_csv(OUT/"ablation_rows.csv", index=False)
    print(ablation.to_string(index=False)); print("D selection:", d_params["selection_status"])
    previous_params_dir = config.PARAMS_DIR
    try:
        config.PARAMS_DIR = PARAMETER_EXPORT
        if not SMOKE and d_params["selection_complete"]:
            print("Formal D parameters:", utils.save_params("D", d_params))
    finally:
        config.PARAMS_DIR = previous_params_dir
    # Required comparison tables. Missing measurement cells remain blank.
    timing_log = json.loads((OUT/"probabilities_summary.json").read_text())
    common_inference = float(np.mean(list(timing_log["inference_seconds"].values())))
    ablation["inference_seconds"] = common_inference
    ablation["total_seconds"] = common_inference + ablation.postprocessing_seconds
    comparison = ablation[ablation.variant.isin(["Baseline", "Filtering + Erosion"])].copy()
    comparison["method"] = ["C" if v == "Baseline" else "D" for v in comparison.variant]
    comparison.to_csv(OUT/"table2_c_vs_d.csv", index=False)
    metrics = ["precision","recall","f1","iou","mean_width_mae","max_width_mae","signed_width_error","max_width_bias","relative_length_error","detection_failures","inference_seconds","postprocessing_seconds"]
    ablation.set_index("variant")[metrics].T.to_csv(OUT/"table3_ablation.csv")
    settings = {"framework":"PyTorch", "python":training["python"], "torch":training["torch"],
        "input_shape":"3 x 256 x 256", "encoder_depth":params["depth"], "base_filters":params["base_channels"],
        "model_parameters":training["model_parameters"], "batch_size":params["batch_size"],
        "learning_rate":params["learning_rate"], "optimizer":"Adam", "loss":"normalized weighted logits BCE + soft Dice",
        "bce_weight":params["bce_weight"], "epochs":len(history), "best_epoch":training["best_epoch"],
        "early_stopping_patience":params["patience"], "hardware":training["hardware"],
        "training_seconds":training["training_seconds"], "probability_threshold":c_params["threshold"],
        "run_kind":RUN_KIND, "tile_overlap_pixels":params["overlap"], "seed":params["seed"]}
    pd.DataFrame(list(settings.items()), columns=["setting","value"]).to_csv(OUT/"table1_network_settings.csv", index=False)
else:
    print("Post-processing disabled; set RUN_POSTPROCESS=True when ready")


## Optional qualitative figures
Set RUN_POSTPROCESS=True only when ready. Formal mode retains all 69 validation images; no test search. Width results remain unavailable while shared measurement is a placeholder.


In [ ]:
if RUN_POSTPROCESS:
    example_ids = val_ids[:min(4, len(val_ids))]
    for image_id in example_ids:
        image, ref = data.load_pair(image_id)
        prob = np.load(CACHE_DIR/f"probabilities/{image_id}_prob.npy", allow_pickle=False)
        mask_c = prob >= c_params["threshold"]
        mask_d = method_d.refine_d(mask_c, d_params)
        utils.save_mask(mask_c, OUT/f"masks/C/{image_id}.png")
        utils.save_mask(mask_d, OUT/f"masks/D/{image_id}.png")
        def overlay(pred):
            canvas = image.astype(np.float32)/255
            for selection, color in [(pred & ref,[0,1,0]),(pred & ~ref,[1,0,0]),(~pred & ref,[0,0,1])]:
                canvas[selection] = .35*canvas[selection]+.65*np.array(color)
            return canvas
        arrays = [image,ref,prob,mask_c,mask_d,overlay(mask_c),overlay(mask_d)]
        titles = ["Original","Ground truth","C probability","C binary","D binary","C overlay","D overlay"]
        for view in ["full", "crop"]:
            if view == "crop" and ref.any():
                rr, cc = np.argwhere(ref)[len(np.argwhere(ref))//2]
                region = (slice(max(0,rr-192),min(ref.shape[0],rr+192)),slice(max(0,cc-192),min(ref.shape[1],cc+192)))
            else:
                region = (slice(None),slice(None))
            fig, axes = plt.subplots(1, 7, figsize=(21,4))
            for ax,a,title in zip(axes,arrays,titles):
                ax.imshow(a[region], cmap="gray", vmin=0 if a.ndim==2 else None, vmax=1 if a.ndim==2 else None)
                ax.set_title(title); ax.axis("off")
            fig.suptitle(f"{RUN_KIND} validation {image_id}: {view}")
            fig.tight_layout(); fig.savefig(OUT/f"figures/qualitative_{image_id}_{view}.png", dpi=200)
            plt.show(); plt.close(fig)
        fig, axes = plt.subplots(1,4,figsize=(12,4))
        for ax,(name,(f,e)) in zip(axes,method_d.ABLATIONS.items()):
            pred = method_d.refine_d(mask_c,{**d_params,"use_filter":f,"use_erosion":e})
            ax.imshow(overlay(pred)[region]);ax.set_title(name+f"\nforeground {pred.mean():.1%}");ax.axis("off")
        fig.suptitle(f"{RUN_KIND} validation {image_id}: D ablation")
        fig.tight_layout();fig.savefig(OUT/f"figures/ablation_{image_id}.png",dpi=250)
        plt.show();plt.close(fig)
else:
    print("Post-processing disabled; set RUN_POSTPROCESS=True when ready")


## Persistent artifacts and final status
Nothing depends on copying this folder at the last cell: OUT already points to Drive in Colab. Probabilities remain under /content and can be regenerated from best.pt; they are not repeatedly uploaded. Keep the run's code snapshot/archive/environment so resume checks can succeed. Never install smoke weights as the team model. Review formal C parameters before copying to params/C.json. D.json is exported only after valid full-validation width selection. Final test remains a separate coordinated team action.


In [ ]:
if (OUT/"training/training.json").is_file():
    training=json.loads((OUT/"training/training.json").read_text(encoding="utf-8"))
    completion={"run_kind":RUN_MODE,"training_status":training["status"],
        "formal_training_completed":RUN_MODE=="formal" and training["status"]=="completed",
        "test_evaluation_completed":False,"postprocess_requested":RUN_POSTPROCESS,
        "persistent_output":str(OUT),"temporary_probabilities":str(CACHE_DIR/"probabilities"),
        "drive_path_used":IN_COLAB,"remote_durability_guaranteed":False}
    method_c._write_json(OUT/"completion.json",completion)
    print(json.dumps(completion,indent=2))
    print("Persistent checkpoints/log/CSV/curves:",OUT/"training")
    print("Copy the complete code snapshot and dataset archive to durable team storage; do not commit model artifacts.")
